<a href="https://colab.research.google.com/github/Thirupathi1356/Computer-Vision-/blob/main/OpticalFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install opencv-python numpy

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow


def calculate_speed(flow, scale_factor, fps):
    """
    Estimate speed in km/h from optical flow vectors
    """
    if flow.size == 0:
        return 0

    mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
    avg_mag = np.mean(mag)

    speed_mps = avg_mag * scale_factor * fps
    speed_kmph = speed_mps * 3.6

    return speed_kmph


def detect_vehicles(fg_mask):
    """
    Detect vehicles using contours
    """
    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vehicles = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area > 1200:   # filter small noise
            x, y, w, h = cv2.boundingRect(cnt)

            aspect_ratio = w / float(h)

            if 0.3 < aspect_ratio < 3.0:
                vehicles.append((x, y, w, h))

    return vehicles


def draw_bounding_box(frame, vehicles, flow, scale_factor, fps):
    """
    Draw bounding boxes and show vehicle speed
    """
    for (x, y, w, h) in vehicles:

        roi_flow = flow[y:y+h, x:x+w]

        speed = calculate_speed(roi_flow, scale_factor, fps)

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        cv2.putText(frame,
                    f"{speed:.2f} km/h",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,255,0),
                    2)


def main(video_path, scale_factor, fps):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video")
        return

    ret, prev_frame = cap.read()

    if not ret:
        print("Error: Cannot read first frame")
        return

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    bg_subtractor = cv2.createBackgroundSubtractorMOG2()

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optical Flow
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Background subtraction
        fg_mask = bg_subtractor.apply(gray)

        # Remove noise
        kernel = np.ones((5,5), np.uint8)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)

        vehicles = detect_vehicles(fg_mask)

        draw_bounding_box(frame, vehicles, flow, scale_factor, fps)

        cv2_imshow(frame)

        prev_gray = gray

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":

    video_path = r"/content/Moving car animation - Jessie Baker (1080p, h264).mp4"

    scale_factor = 0.05
    fps = 30

    main(video_path, scale_factor, fps)